# Staged Multi-Molecule SFT On Colab

This notebook is the Colab counterpart to the Kaggle multi-molecule SFT notebook, but it uses the new Colab bootstrap flow. It bootstraps the repo in `/content/Thesis`, then delegates dependency installation, mini post-training dataset download, and training to `scripts/init_colab.py`.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/content/Thesis")

%cd /content
if (REPO_DIR / ".git").exists():
    print(f"Reusing {REPO_DIR}")
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", REPO_URL, REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print({"repo_dir": str(REPO_DIR), "repo_branch": REPO_BRANCH})


In [ ]:
DATASET_MODE = "auto"
TRAIN_DATASET_FILE_ID = "1fwRIHrcq0nGA1OJCCWqdxvGgcbW2oKY3"
CONFIG_OVERRIDE = None
EXTRA_SCRIPT_ARGS = []

OUTPUT_DIR = REPO_DIR / "outputs" / "multi_molecule_sft"
BEST_CHECKPOINT = OUTPUT_DIR / "checkpoints" / "best"
RUN_SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"
MINI_DATASET_DIR = REPO_DIR / "data" / "mini_post_training"
MINI_DATASET_CHECKS = {
    "train_multimol": MINI_DATASET_DIR / "post_training_processed" / "train_multimol.jsonl",
    "split_train": MINI_DATASET_DIR / "grouped_splits" / "train_multimol.jsonl",
    "split_validation": MINI_DATASET_DIR / "grouped_splits" / "validation_multimol.jsonl",
    "split_test": MINI_DATASET_DIR / "grouped_splits" / "test_multimol.jsonl",
}


In [ ]:
%cd {REPO_DIR}
command = [
    sys.executable,
    "scripts/init_colab.py",
    "--stage",
    "multi_sft",
    "--repo-url",
    REPO_URL,
    "--repo-branch",
    REPO_BRANCH,
    "--repo-dir",
    str(REPO_DIR),
    "--dataset-mode",
    DATASET_MODE,
    "--train-dataset-file-id",
    TRAIN_DATASET_FILE_ID,
]
if CONFIG_OVERRIDE:
    command.extend(["--config", str(CONFIG_OVERRIDE)])
if EXTRA_SCRIPT_ARGS:
    command.append("--extra-script-args")
    command.extend(EXTRA_SCRIPT_ARGS)

print("Running:", " ".join(command))
subprocess.run(command, check=True)


In [ ]:
print({
    "output_dir": str(OUTPUT_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
    "best_checkpoint": str(BEST_CHECKPOINT),
    "best_checkpoint_exists": BEST_CHECKPOINT.exists(),
    "run_summary": str(RUN_SUMMARY_PATH),
    "run_summary_exists": RUN_SUMMARY_PATH.exists(),
    "mini_dataset_dir": str(MINI_DATASET_DIR),
    "mini_dataset_exists": MINI_DATASET_DIR.exists(),
    "mini_dataset_checks": {name: path.exists() for name, path in MINI_DATASET_CHECKS.items()},
})
